# Classical and Deep Learning Approaches to Image Classification

**Course:** CMSC 422 — Machine Learning  
**Author:** Kiyana Amirian

---

## Overview

This notebook benchmarks a progression of classification methods on two datasets:

- **MNIST** (handwritten digits, 10 classes): evaluated with SVMs, dimensionality reduction (PCA/LDA), and logistic regression (sklearn and from scratch)
- **Monkey species** (10 classes): evaluated with VGG19 transfer learning (frozen backbone and fine-tuned)

### Methods covered

| Section | Method | Dataset |
|---------|--------|---------|
| 1 | SVM (linear, RBF, polynomial) on raw features | MNIST |
| 2 | SVM + PCA / LDA dimensionality reduction | MNIST |
| 3 | Logistic regression (scikit-learn) | MNIST |
| 4 | Logistic regression from scratch (NumPy softmax + cross-entropy) | MNIST |
| 5 | Custom CNN | MNIST |
| 6 | VGG19 transfer learning (frozen → fine-tuned) | Monkey species |

### Data requirements

- **MNIST**: loaded automatically via `tensorflow.keras.datasets`
- **Monkey species**: place in `./data/training/training/` and `./data/validation/validation/` (10 class subdirectories each)


In [ ]:

from tensorflow.keras.datasets import mnist
import matplotlib.pyplot as plt

(train_X, train_y), (test_X, test_y) = mnist.load_data()
# Select 12 sample images from train_X
sample_indices = range(12)  # or use any custom indices
samples = [train_X[i] for i in sample_indices]

# Create 2x6 subplot grid
plt.figure(figsize=(12, 4))  # adjust size as needed
for i, img in enumerate(samples):
    plt.subplot(2, 6, i + 1)
    plt.imshow(img.squeeze())  # .squeeze() handles (28, 28, 1)
    plt.axis('off')
    plt.title(f"Index {sample_indices[i]}")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create side-by-side subplots
plt.figure(figsize=(12, 5))

# Left subplot: train labels
plt.subplot(1, 2, 1)
sns.countplot(x=train_y)
plt.title("Train Labels Distribution")
plt.xlabel("Class")
plt.ylabel("Count")

# Right subplot: test labels
plt.subplot(1, 2, 2)
sns.countplot(x=test_y)
plt.title("Test Labels Distribution")
plt.xlabel("Class")
plt.ylabel("Count")

plt.tight_layout()
plt.show()


## 1. Support Vector Machines (Raw Features)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
import seaborn as sns
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from array import array

def main(): 
    (train_X, train_y), (test_X, test_y) = mnist.load_data()
    
    # Display a sample image
    sample = train_X[95, :, :]
    plt.figure(1)
    plt.imshow(sample)
    
    # Reshape images to vectors
    train_X = train_X.reshape((train_X.shape[0], -1))
    train_X = GetSpacedElements(train_X, 10000)
    train_y = GetSpacedElements(train_y, 10000)
    test_X = test_X.reshape((test_X.shape[0], -1))
    
    # Plot label histograms
    plt.figure(2)
    sns.countplot(x=train_y)
    
    
    plt.title("Histogram of labels in train dataset subset")
    plt.figure(3)
    sns.countplot(x=test_y)
    plt.title("Histogram of labels in test dataset")
    
    # SVM with different kernels using scikit-learn
    kernels = ['linear', 'poly', 'rbf']
    for kernel in kernels:
        model = SVC(kernel=kernel)
        model.fit(train_X, train_y)
        predictions = model.predict(test_X)
        accuracy = accuracy_score(test_y, predictions)
        print(f"{kernel.capitalize()} kernel SVM accuracy: {accuracy:.4f}")

def GetSpacedElements(array, numElems):
    if len(array.shape) == 2:
        out = array[np.round(np.linspace(0, array.shape[0]-1, numElems)).astype(int), :]
    else:
        out = array[np.round(np.linspace(0, array.shape[0]-1, numElems)).astype(int)]    
    return out

if __name__ == "__main__": 
    main()


## 2. SVMs with Normalization, PCA, and LDA


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.datasets import mnist

def main():
    # Load MNIST
    (train_X, train_y), (test_X, test_y) = mnist.load_data()

    # Display a sample image
    plt.figure()
    plt.imshow(train_X[95], cmap='gray')
    plt.title("Sample Image")

    # Flatten images
    train_X = train_X.reshape((train_X.shape[0], -1)).astype("float32") / 255.0
    test_X = test_X.reshape((test_X.shape[0], -1)).astype("float32") / 255.0

    # Standardize data
    scaler = StandardScaler()
    train_X = scaler.fit_transform(train_X)
    test_X = scaler.transform(test_X)

    # Plot histograms
    plt.figure()
    sns.countplot(x=train_y)
    plt.title("Train label histogram")

    plt.figure()
    sns.countplot(x=test_y)
    plt.title("Test label histogram")

    # Kernels to use
    kernels = ['linear', 'poly', 'rbf']

    # =========================
    # SVM on Raw Data
    # =========================
    print("\n🔹 SVM on raw 784 features")
    for kernel in kernels:
        print(f"\n▶ Raw + {kernel} kernel")
        model = SVC(kernel=kernel, gamma='scale')
        start = time.time()
        model.fit(train_X, train_y)
        print(f"Training time: {time.time() - start:.2f} sec")
        predictions = model.predict(test_X)
        acc = accuracy_score(test_y, predictions)
        print(f"✅ Final Accuracy: {acc:.4f}")

    # =========================
    # PCA + SVM
    # =========================
    print("\n🔹 SVM after PCA (50 components)")
    pca = PCA(n_components=50)
    x_train_pca = pca.fit_transform(train_X)
    x_test_pca = pca.transform(test_X)

    for kernel in kernels:
        print(f"\n▶ PCA + {kernel} kernel")
        model = SVC(kernel=kernel, gamma='scale')
        start = time.time()
        model.fit(x_train_pca, train_y)
        print(f"Training time: {time.time() - start:.2f} sec")
        predictions = model.predict(x_test_pca)
        acc = accuracy_score(test_y, predictions)
        print(f"Accuracy: {acc:.4f}")
        print(classification_report(test_y, predictions, digits=4))

    # =========================
    # LDA + SVM
    # =========================
    print("\n🔹 SVM after LDA (9 components)")
    lda = LDA(n_components=9)
    x_train_lda = lda.fit_transform(train_X, train_y)
    x_test_lda = lda.transform(test_X)

    for kernel in kernels:
        print(f"\n▶ LDA + {kernel} kernel")
        model = SVC(kernel=kernel, gamma='scale')
        start = time.time()
        model.fit(x_train_lda, train_y)
        print(f"Training time: {time.time() - start:.2f} sec")
        predictions = model.predict(x_test_lda)
        acc = accuracy_score(test_y, predictions)
        print(f"Accuracy: {acc:.4f}")
        print(classification_report(test_y, predictions, digits=4))

if __name__ == "__main__":
    main()


## 3. Logistic Regression (scikit-learn)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from keras.datasets import mnist
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.linear_model import LogisticRegression

def main(): 
    (train_X, train_y), (test_X, test_y) = mnist.load_data()
    
    "Testing one sample data"
    sample = train_X[8, :, :];
    plt.figure(1)
    plt.imshow(sample)
    
    "Reshaping the images into vectors"
    train_X = train_X.reshape((train_X.shape[0], train_X.shape[1]*train_X.shape[2]))
    train_X = GetSpacedElements(train_X, 10000)
    train_y = GetSpacedElements(train_y, 10000)
    test_X = test_X.reshape((test_X.shape[0], test_X.shape[1]*test_X.shape[2]))

    "Evaluating histogram of labels"
    plt.figure(2)
    sns.countplot(x=train_y)
    plt.title("Histogram of labels in train dataset subset")
    plt.figure(3)
    sns.countplot(x=test_y)
    plt.title("Histogram of labels in test dataset")
    
    "PCA and LDA"
    pca=PCA(n_components=50)
#     pca = PCA(train_X.shape[1])
    train_X_PCA = pca.fit_transform(train_X)
    test_X_PCA = pca.transform(test_X)
    
    lda = LDA(n_components = 9)
    train_X_LDA = lda.fit_transform(train_X, train_y)
    test_X_LDA = lda.transform(test_X)

    "Logistic Regression"
    clf = LogisticRegression(C=50.0 / 10000, penalty="l1", solver="saga", tol=0.1)
    clf.fit(train_X, train_y)
    score = clf.score(test_X, test_y)
    print("Logistic Regression score:", score*100)
    
    clf = LogisticRegression(C=50.0 / 10000, penalty="l1", solver="saga", tol=0.1)
    clf.fit(train_X_PCA, train_y)
    score_PCA = clf.score(test_X_PCA, test_y)
    print("Logistic Regression after PCA score:", score_PCA*100)
    
    clf = LogisticRegression(C=50.0 / 10000, penalty="l1", solver="saga", tol=0.1)
    clf.fit(train_X_LDA, train_y)
    score_LDA = clf.score(test_X_LDA, test_y)
    print("Logistic Regression after LDA score:", score_LDA*100)
      

def GetSpacedElements(array, numElems):
    if len(array.shape) == 2:
        out = array[np.round(np.linspace(0, array.shape[0]-1, numElems)).astype(int), :]
    else:
        out = array[np.round(np.linspace(0, array.shape[0]-1, numElems)).astype(int)]    
    return out

if __name__ == "__main__": 
    main() 

## 4. Logistic Regression from Scratch (NumPy)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import accuracy_score


def GetSpacedElements(array, numElems):
    if len(array.shape) == 2:
        indices = np.round(np.linspace(0, array.shape[0] - 1, numElems)).astype(int)
        return array[indices, :]
    else:
        indices = np.round(np.linspace(0, array.shape[0] - 1, numElems)).astype(int)
        return array[indices]


# Load MNIST from Keras
(train_X, train_y), (test_X, test_y) = mnist.load_data()

# Flatten and normalize
train_X = train_X.reshape((train_X.shape[0], -1)) / 255.0
test_X = test_X.reshape((test_X.shape[0], -1)) / 255.0

# Subsample using evenly spaced sampling
subset = 10000
train_X = GetSpacedElements(train_X, subset)
train_y = GetSpacedElements(train_y, subset)

# One-hot encode
encoder = OneHotEncoder(sparse_output=False)
train_y_onehot = encoder.fit_transform(train_y.reshape(-1, 1))
test_y_onehot = encoder.transform(test_y.reshape(-1, 1))

# Softmax function
def softmax(z):
    z -= np.max(z, axis=1, keepdims=True)  # for numerical stability
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

# Cross-entropy loss
def cross_entropy(y_true, y_pred):
    return -np.mean(np.sum(y_true * np.log(y_pred + 1e-8), axis=1))

# Gradient computation
def compute_gradient(X, y_true, y_pred):
    return np.dot((y_pred - y_true).T, X) / X.shape[0]

# Training and evaluation function
def train_logistic_regression(X_train, y_train_oh, X_test, y_test, epochs=100, lr=0.1):
    num_classes = y_train_oh.shape[1]
    num_features = X_train.shape[1]
    W = np.random.randn(num_classes, num_features) * 0.01
    loss_history = []

    for epoch in range(epochs):
        logits = np.dot(X_train, W.T)
        probs = softmax(logits)
        loss = cross_entropy(y_train_oh, probs)
        grad = compute_gradient(X_train, y_train_oh, probs)
        W -= lr * grad
        loss_history.append(loss)

    def predict(X):
        return np.argmax(np.dot(X, W.T), axis=1)

    test_preds = predict(X_test)
    acc = accuracy_score(test_y, test_preds)
    return acc, loss_history

# Baseline on raw input (already normalized by /255.0 but not standardized)
print("Training on raw input (no standardization)...")
acc_base, loss_base = train_logistic_regression(train_X, train_y_onehot, test_X, test_y)

# Now standardize for PCA/LDA
scaler = StandardScaler()
train_X_scaled = scaler.fit_transform(train_X)
test_X_scaled = scaler.transform(test_X)

# PCA
pca = PCA(n_components=50)
train_X_pca = pca.fit_transform(train_X_scaled)
test_X_pca = pca.transform(test_X_scaled)
print("Training with PCA (50 components)...")
acc_pca, loss_pca = train_logistic_regression(train_X_pca, train_y_onehot, test_X_pca, test_y)

# LDA (9 components max for 10 classes)
lda = LDA(n_components=9)
train_X_lda = lda.fit_transform(train_X_scaled, train_y)
test_X_lda = lda.transform(test_X_scaled)
print("Training with LDA (9 components)...")
acc_lda, loss_lda = train_logistic_regression(train_X_lda, train_y_onehot, test_X_lda, test_y)

# Plotting
plt.plot(loss_base, label="Baseline")
plt.plot(loss_pca, label="PCA (50)")
plt.plot(loss_lda, label="LDA (9)")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("Training Loss Over Epochs")
plt.legend()
plt.grid(True)
plt.show()

# Final Accuracy
print(f"\nFinal Test Accuracies:")
print(f"Baseline: {acc_base * 100:.2f}%")
print(f"PCA     : {acc_pca * 100:.2f}%")
print(f"LDA     : {acc_lda * 100:.2f}%")


## 5. Convolutional Neural Network (MNIST)


In [ ]:
import keras
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from keras.datasets import mnist
from keras.layers import Dense, Activation, Flatten, Conv2D, MaxPooling2D
from keras.models import Sequential
from keras.utils import to_categorical


def main(): 
    (train_X, train_y), (test_X, test_y) = mnist.load_data()


    # Reshape and downsample training data
    train_X = train_X.reshape((train_X.shape[0], -1))
    train_X = GetSpacedElements(train_X, 10000)
    train_X = train_X.reshape(-1, 28, 28, 1)
    train_y = GetSpacedElements(train_y, 10000)

    # Reshape test data
    test_X = test_X.reshape(-1, 28, 28, 1)

    # One-hot encoding of labels
    train_Y_one_hot = to_categorical(train_y)
    test_Y_one_hot = to_categorical(test_y)

    # Define CNN model
    model = Sequential()
    model.add(Conv2D(64, (3,3), input_shape=(28, 28, 1)))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2,2)))

    model.add(Conv2D(64, (3,3)))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2,2)))

    model.add(Conv2D(64, (3,3)))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2,2)))

    model.add(Flatten())
    model.add(Dense(64))
    model.add(Dense(10))
    model.add(Activation('softmax'))

    model.compile(
        loss=keras.losses.categorical_crossentropy,
        optimizer=keras.optimizers.Adam(),
        metrics=['accuracy']
    )

    print(model.summary())

    # Train model and track history
    history = model.fit(
        train_X, train_Y_one_hot,
        batch_size=64,
        epochs=5,
        validation_split=0.2,
        verbose=1
    )

    # Evaluate on test set
    test_loss, test_acc = model.evaluate(test_X, test_Y_one_hot, verbose=0)
    print(f"\nFinal Test Accuracy: {test_acc * 100:.2f}%")
    print(f"Final Test Loss: {test_loss:.4f}")

    # Plot accuracy
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Accuracy over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # Plot loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Loss over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()


def GetSpacedElements(array, numElems):
    indices = np.round(np.linspace(0, array.shape[0] - 1, numElems)).astype(int)
    return array[indices] if len(array.shape) == 1 else array[indices, :]


if __name__ == "__main__": 
    main()


## 6. Transfer Learning with VGG19 (Monkey Species)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import preprocess_input
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.utils import plot_model

# 📁 Step 0: Data directories
train_dir = "./data/training/training/"
val_dir = "./data/validation/validation/"

# 🔄 Step 1: Image generators with preprocessing and augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=4,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=4,
    class_mode='categorical'
)

# 🧱 Step 2: Define custom CNN model
def create_custom_cnn(input_shape=(224, 224, 3), num_classes=10):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(120, activation='relu'),
        Dense(84, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    return model

# 🧠 Step 3: Train custom CNN
print("\n📦 Training custom CNN model...")
model = create_custom_cnn(num_classes=train_generator.num_classes)
model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
plot_model(model, to_file='custom_cnn.png', show_shapes=True)
model.summary()

start = time.time()
history_cnn = model.fit(train_generator, epochs=5, validation_data=val_generator)
print("🕒 Training time (custom CNN):", round(time.time() - start), "sec")
model.save('custom_cnn_model.h5')

# 🧪 Evaluate custom CNN
loss_custom, acc_custom = model.evaluate(val_generator, verbose=0)
print(f"📊 Custom CNN Accuracy       : {acc_custom * 100:.2f}%")

# 🔁 Step 4: Transfer learning with VGG19
print("\n📦 Training VGG19 Transfer Learning model...")
base_model = VGG19(include_top=False, weights='imagenet', input_shape=(224, 224, 3), pooling='avg')
base_model.trainable = False  # Freeze all conv layers

# Add new fully connected head
x = base_model.output
x = Dense(1024, activation='relu')(x)
x = Dropout(0.4)(x)
output = Dense(train_generator.num_classes, activation='softmax')(x)

vgg_model = Model(inputs=base_model.input, outputs=output)
vgg_model.compile(optimizer=SGD(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
plot_model(vgg_model, to_file='vgg_model.png', show_shapes=True)
vgg_model.summary()

start = time.time()
history_vgg = vgg_model.fit(train_generator, epochs=5, validation_data=val_generator)
print("🕒 Training time (VGG19 frozen):", round(time.time() - start), "sec")
vgg_model.save('vgg19_transfer_model.h5')

# 🧪 Evaluate frozen VGG19 model
loss_vgg, acc_vgg = vgg_model.evaluate(val_generator, verbose=0)
print(f"📊 VGG19 Transfer Accuracy    : {acc_vgg * 100:.2f}%")

# 🧠 Step 5 (Optional): Fine-tune VGG19
print("\n🔧 Fine-tuning VGG19 model (unfreezing all layers)...")
base_model.trainable = True

vgg_model.compile(optimizer=SGD(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
history_fine = vgg_model.fit(train_generator, epochs=5, validation_data=val_generator)
vgg_model.save('vgg19_finetuned_model.h5')

# 🧪 Evaluate fine-tuned VGG19 model
loss_fine, acc_fine = vgg_model.evaluate(val_generator, verbose=0)
print(f"📊 Fine-tuned VGG19 Accuracy  : {acc_fine * 100:.2f}%")

# 🏁 Final comparison
print("\n📈 Final Accuracy Comparison:")
print(f"   Custom CNN Accuracy        : {acc_custom * 100:.2f}%")
print(f"   VGG19 Transfer Accuracy    : {acc_vgg * 100:.2f}%")
print(f"   Fine-tuned VGG19 Accuracy  : {acc_fine * 100:.2f}%")

# Combine frozen and fine-tuned VGG history
history_vgg_full = {
    'accuracy': history_vgg.history['accuracy'] + history_fine.history['accuracy'],
    'val_accuracy': history_vgg.history['val_accuracy'] + history_fine.history['val_accuracy']
}



import matplotlib.pyplot as plt

# === 1. Combined Plot: All Models ===
plt.figure(figsize=(10, 6))
plt.plot(history_cnn.history['accuracy'], label='Custom CNN - Train')
plt.plot(history_cnn.history['val_accuracy'], label='Custom CNN - Val')
plt.plot(history_vgg.history['accuracy'], label='VGG19 Frozen - Train')
plt.plot(history_vgg.history['val_accuracy'], label='VGG19 Frozen - Val')
plt.plot(history_fine.history['accuracy'], label='VGG19 Fine-tuned - Train')
plt.plot(history_fine.history['val_accuracy'], label='VGG19 Fine-tuned - Val')
plt.title("Training and Validation Accuracy for All Models")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.savefig("all_models_accuracy.png")
plt.show()

# === 2. Separate Plot: Custom CNN ===
plt.figure(figsize=(6, 4))
plt.plot(history_cnn.history['accuracy'], label='Train')
plt.plot(history_cnn.history['val_accuracy'], label='Validation')
plt.title("Custom CNN Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.savefig("custom_cnn_accuracy.png")
plt.show()

# === 3. Separate Plot: VGG19 Frozen ===
plt.figure(figsize=(6, 4))
plt.plot(history_vgg.history['accuracy'], label='Train')
plt.plot(history_vgg.history['val_accuracy'], label='Validation')
plt.title("VGG19 Frozen Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.savefig("vgg19_frozen_accuracy.png")
plt.show()

# === 4. Separate Plot: VGG19 Fine-tuned ===
plt.figure(figsize=(6, 4))
plt.plot(history_fine.history['accuracy'], label='Train')
plt.plot(history_fine.history['val_accuracy'], label='Validation')
plt.title("VGG19 Fine-tuned Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.savefig("vgg19_finetuned_accuracy.png")
plt.show()

